In [4]:
import copy
import torch 
import pandas as pd
import numpy as np
import pickle as pkl

from pathlib import Path
from tqdm import tqdm 

from neuralhydrology.evaluation import get_tester
from neuralhydrology.utils.config import Config 
from neuralhydrology.datautils.utils import load_scaler

import neuralhydrology as nh

In [5]:
scaler = load_scaler(Path("/home/wuhlmann/BA/test_runs/runs/full_q_512_3011_185525"))

In [19]:
basin_attr = pd.read_table("/home/wuhlmann/BA/data/raw_data/2_LamaH-CE_daily/B_basins_intermediate_all/1_attributes/Catchment_attributes.csv", header=0, sep=";")

In [6]:
# Load the config.
run_dir_path = Path("/home/wuhlmann/BA/test_runs/runs/full_q_512_3011_185525")
cfg = Config(run_dir_path / "config.yml")


In [7]:
#attribute one. 
tester = get_tester(cfg=cfg, run_dir=run_dir_path, period="test", init_model=True)


In [8]:
raw_results = tester.evaluate(save_results=False, metrics=["NSE"])

# Evaluation: 100%|██████████| 88/88 [01:44<00:00,  1.19s/it]


In [18]:
for i, attr in enumerate(sorted(cfg.static_attributes)):
    
	mu = scaler["attribute_means"][attr]
	s = scaler["attribute_means"][attr]

	attr_normalized_value = tester.cached_datasets["116"]._attributes["116"][i]

	print(attr_normalized_value + 0.1 == attr_normalized_value + (s*0.1)/s)

tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)
tensor(True)


In [6]:
with open("/home/wuhlmann/BA/data/processed_data/SA/full_q_512_3011_185525_raw_nse_deltas.p", "rb") as f: 
    raw_nse = pkl.load(f)

with open("/home/wuhlmann/BA/data/processed_data/SA/full_q_512_3011_185525_weighted_means_deltas.p", "rb") as f: 
    means = pkl.load(f)

In [7]:
attr_counter = {}

for df in means.values(): 
    
	top_attr = df.index[0]

	if not top_attr in attr_counter: 
		attr_counter[top_attr] = 0
	attr_counter[top_attr] += 1

In [8]:
attr_counter

{'elev_mean': 36, 'area_calc': 41, 'elev_ran': 11}

In [98]:
sorted(cfg.static_attributes)

['agr_fra',
 'area_calc',
 'arid_1',
 'bare_fra',
 'bedrk_dep',
 'clay_fra',
 'elev_mean',
 'elev_ran',
 'et0_mean',
 'forest_fra',
 'frac_snow',
 'glac_fra',
 'gvf_diff',
 'gvf_max',
 'hi_prec_du',
 'hi_prec_fr',
 'lai_diff',
 'lai_max',
 'lake_fra',
 'lo_prec_du',
 'lo_prec_fr',
 'p_mean',
 'p_season',
 'sand_fra',
 'silt_fra',
 'slope_mean',
 'soil_condu',
 'soil_poros',
 'urban_fra']

In [12]:
tester.cached_datasets["116"]._attributes

{'116': tensor([-1.1202, -0.6565, -1.0047, -0.3082, -0.5849, -0.6791,  1.2046,  0.6526,
         -0.6400, -0.0641,  1.2596, -0.1595,  2.0378,  0.5849, -1.6204, -0.9843,
         -0.4451, -0.5979, -0.2477, -1.1745, -0.9841,  1.1747,  0.5695,  0.5160,
         -0.3012,  0.9772, -0.9775, -0.3035, -0.6967])}

In [13]:
means = []

for id in tester.cached_datasets.keys():

	m = torch.mean(tester.cached_datasets[id]._attributes[id])
	print(torch.max(tester.cached_datasets[id]._attributes[id]))
	means.append(m)

tensor(2.0378)
tensor(1.8929)
tensor(1.9653)
tensor(1.9284)
tensor(3.9551)
tensor(1.7279)
tensor(13.5308)
tensor(2.1793)
tensor(1.0623)
tensor(2.0082)
tensor(1.7507)
tensor(4.8683)
tensor(1.9776)
tensor(2.3079)
tensor(2.1605)
tensor(1.9200)
tensor(5.7248)
tensor(1.6213)
tensor(2.6263)
tensor(2.3138)
tensor(1.5036)
tensor(1.1647)
tensor(3.8201)
tensor(2.4080)
tensor(1.6072)
tensor(1.6657)
tensor(2.0096)
tensor(3.2528)
tensor(2.1703)
tensor(1.9619)
tensor(1.5416)
tensor(2.0910)
tensor(3.8518)
tensor(1.4223)
tensor(2.0511)
tensor(5.1063)
tensor(1.3861)
tensor(1.4884)
tensor(2.4723)
tensor(1.8316)
tensor(1.7464)
tensor(5.4114)
tensor(2.4371)
tensor(1.9604)
tensor(2.5923)
tensor(1.5036)
tensor(3.0944)
tensor(2.1961)
tensor(2.1640)
tensor(3.1782)
tensor(2.2292)
tensor(1.7490)
tensor(1.9873)
tensor(1.3930)
tensor(1.1290)
tensor(1.6179)
tensor(1.7303)
tensor(5.6641)
tensor(3.7975)
tensor(5.9639)
tensor(2.6171)
tensor(3.0389)
tensor(3.9904)
tensor(3.8896)
tensor(2.4985)
tensor(4.0560)
tensor(1.

In [42]:
from numpy.random import Generator, MT19937

In [ ]:
basin_ids_list = list(tester.cached_datasets.keys())
baseline_nse_values = np.array([raw_results[id]["1D"]["NSE"] for id in basin_ids_list])
attr_ids = [0]

rng = Generator(MT19937(1277)) 



noise_amounts = []

for i in range(10):

	noise_amounts.append(rng.uniform(-0.1, 0.1))

print(noise_amounts)

num_basins = len(basin_ids_list)
num_attr = len(attr_ids)
noise_levels = len(noise_amounts)

# 3D array with dim basin x attribute x noise_level, to hold raw numeric values
raw_array = np.zeros([num_basins, num_attr, noise_levels])

for attr_id in attr_ids:
	
	for noise_id in range(noise_levels): 

		# restore original attributes values in tester
		tester_copy = copy.deepcopy(tester)

		# add noise to the attribute in every catchment
		for id in basin_ids_list:	
			tester_copy.cached_datasets[id]._attributes[id][attr_id] += noise_amounts[noise_id]

		# run evaluation for the currrent noise level
		tester_result_dict = tester_copy.evaluate(save_results=False, metrics=["NSE"])
		nse_values = np.array([tester_result_dict[id]["1D"]["NSE"] for id in basin_ids_list])

		# write NSE for all basins, for the current attribute and noise level	
		raw_array[:, attr_id, noise_id] = abs(nse_values - baseline_nse_values)

[0.05487206613383655, -0.03649681719980631, -0.08702611202112667, 0.07980325736717062, 0.07339916425369106, -0.08947411646554036, 0.03585066431543041, 0.06494898993347986, -0.03146590737961728, 0.04499257226756978]


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x14e61ace7340>>
Traceback (most recent call last):
  File "/storage/vast-gfz-hpc-01/home/wuhlmann/miniforge3/envs/lamah-ce_lstm/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


# Evaluation:   9%|▉         | 8/88 [00:08<01:29,  1.12s/it]

# Evaluation:   9%|▉         | 8/88 [00:09<01:33,  1.17s/it]


KeyboardInterrupt: 

In [ ]:
results_dict = {}

for i in range(len(basin_ids_list)): 
	
	results_dict[basin_ids_list[i]] = pd.DataFrame(data=raw_array[i,:,:], index=[cfg.static_attributes[0]], columns=noise_amounts)
	

In [ ]:
test_df = results_dict["116"]
test_df

In [ ]:
attr_ranks = dict(zip(test_df.index, [0]*len(test_df.index)))

attr_weights = [0.75, 1, 1, 0.75]

for col, weight in zip(test_df.columns, attr_weights):
	
	order = list(test_df.sort_values(by=col, ascending=False)[col].index)

	for attr in test_df.index: 

		attr_ranks[attr] += (order.index(attr)+1) * weight



In [ ]:
for id in results_dict.keys():

	print(f"{id} {'-'*10}")

	basin_df = results_dict[id]

	attr_ranks = pd.DataFrame(data=[0]*len(basin_df.index), index=basin_df.index, columns=["rank"], dtype="float")

	attr_weights = [0.75, 1, 1, 0.75]

	for col, weight in zip(basin_df.columns, attr_weights):
		
		order = list(basin_df.sort_values(by=col, ascending=False)[col].index)

		for attr in basin_df.index: 

			attr_ranks.loc[attr, "rank"] += (order.index(attr)+1) * weight

	mean_attr_ranks = attr_ranks.apply(lambda x: x/4)

	print(mean_attr_ranks.sort_values(by="rank", ascending=True))
	